In [15]:
# import sys
# !{sys.executable} -m pip install --user -qq scikit-learn xgboost

In [2]:
!pip install -qq pandas numpy matplotlib seaborn 
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb
import pandas as pd
import numpy as np
%matplotlib inline

In [3]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
car_fuel_eff = pd.read_csv(url)
df_fe = car_fuel_eff
df_fe.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [4]:
df_fe.info()
df_fe.columns = df_fe.columns.str.lower().str.replace(' ', '_')

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   model_year           10000 non-null  int64  
 1   origin               10000 non-null  str    
 2   fuel_type            10000 non-null  str    
 3   drivetrain           10000 non-null  str    
 4   num_doors            10000 non-null  int64  
 5   engine_displacement  10000 non-null  int64  
 6   num_cylinders        10000 non-null  int64  
 7   horsepower           9123 non-null   float64
 8   vehicle_weight       10000 non-null  int64  
 9   acceleration         9736 non-null   float64
 10  fuel_efficiency_mpg  10000 non-null  float64
dtypes: float64(3), int64(5), str(3)
memory usage: 859.5 KB


In [5]:
numerical_cols = df_fe.select_dtypes(include=['float64']).columns
df_fe[numerical_cols] = df_fe[numerical_cols].fillna(0.0)
df_fe.isnull().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [6]:
df = df_fe
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']

train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

In [7]:
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)
feature_names = dv.get_feature_names_out()
feature_index = dt.tree_.feature[0]
split_feature_onehot = feature_names[feature_index]
print(f"1. One-hot or original feature used for splitting: {split_feature_onehot}")


1. One-hot or original feature used for splitting: model_year


In [8]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
print(f"2. RMSE on Validation: {rmse:.4f}")


2. RMSE on Validation: 1.8332


In [9]:
n_estimators_list = [10, 50, 100, 150]
lv_results = {}

for n in n_estimators_list:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    lv_results[n] = round(rmse, 3)
    print(f"n_estimators= {n}: RMSE = {rmse:.3f}")

best_n = min(lv_results, key=lv_results.get)
print("-----------------------------------------------")
print(f"3. Lowest RMSE at n_estimators = {best_n}\n")
print("-----------------------------------------------")

n_estimators= 10: RMSE = 1.833
n_estimators= 50: RMSE = 1.774
n_estimators= 100: RMSE = 1.770
n_estimators= 150: RMSE = 1.769
-----------------------------------------------
3. Lowest RMSE at n_estimators = 150

-----------------------------------------------


In [10]:
max_depths = [10, 15, 20, 25]
depth_scores = {}

for depth in max_depths:
    rmses = []
    for n in n_estimators_list:
        rf = RandomForestRegressor(max_depth=depth, n_estimators=n, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        rmses.append(rmse)

    mean_rmse = np.mean(rmses)
    depth_scores[depth] = mean_rmse
    print(f"max_depth={depth}: Mean RMSE = {mean_rmse:.4f}")

best_depth = min(depth_scores, key=depth_scores.get)
print("-----------------------------------------------")
print(f"4. Best max_depth by mean RMSE = {best_depth}")
print("-----------------------------------------------")

max_depth=10: Mean RMSE = 1.7544
max_depth=15: Mean RMSE = 1.7842
max_depth=20: Mean RMSE = 1.7868
max_depth=25: Mean RMSE = 1.7863
-----------------------------------------------
4. Best max_depth by mean RMSE = 10
-----------------------------------------------


In [11]:
rf_fi = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf_fi.fit(X_train, y_train)
feature_names = dv.get_feature_names_out()
importances = rf_fi.feature_importances_
importance_dict = {}
for name, importance in zip(feature_names, importances):
    orig_name = name.split('=')[0]
    importance_dict[orig_name] = importance_dict.get(orig_name, 0) + importance

target_features = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
for feat in target_features:
    print(f"{feat}: {importance_dict.get(feat, 0):.4f}")

most_important = max(target_features, key=lambda f: importance_dict.get(f, 0))
print("-----------------------------------------------")
print(f"5. Most important feature: {most_important}\n")
print("-----------------------------------------------")

vehicle_weight: 0.2033
horsepower: 0.0783
acceleration: 0.0550
engine_displacement: 0.0613
-----------------------------------------------
5. Most important feature: vehicle_weight

-----------------------------------------------


In [14]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, 'train'), (dval, 'val')]
eta_results = {}

for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 0,
    }
    
    model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    y_pred_xgb = model.predict(dval)
    val_rmse = root_mean_squared_error(y_val, y_pred_xgb)
    eta_results[eta] = val_rmse
    print(f"eta={eta}: Validation RMSE = {val_rmse:.4f}")
best_eta = min(eta_results, key=eta_results.get)
print("-----------------------------------------------")
print(f"6. Best eta or Estimated Target Acceleration(With lowest RMSE): "
      f"{best_eta} (with RMSE = {eta_results[best_eta]:.4f})")
print("-----------------------------------------------")

eta=0.3: Validation RMSE = 1.8265
eta=0.1: Validation RMSE = 1.7248
-----------------------------------------------
6. Best eta or Estimated Target Acceleration(With lowest RMSE): 0.1 (with RMSE = 1.7248)
-----------------------------------------------
